# Blog figures — personal post (BLOG_DRAFT.md)

Every producible `[B#]` asset for the personal post. Runs on the workspace
against `$FM_BASE` (default `/mnt/user_storage/transaction-fm-v2`); every
loader skips loudly if its artifact is missing. Outputs land in the repo's `figures/` dir (commit them — the blogs embed
these paths directly); `$FM_FIG_OUT` overrides. PNG (200dpi) + SVG.

| cell | asset | source artifact |
|---|---|---|
| hero | **B1** two-panel dot+CI | `downstream/<sc>/bootstrap_ci.json` (2048 = 20-ep dir) + `<sc>_fulltest/` |
| arch | **B2** pipeline graphic (plain) | drawn here |
| fulltest | **B7** table-2 dot+CI | fulltest CI jsons + paired bootstrap json |
| reco | **B-RECO-FIG** slices bars | `full_nextmerchant/probe_metrics_v3.json` |
| burst | **B8** ledger + **B9** cumulative curve | `raw/full/transactions.parquet` (computed here) |
| canary | **B16** month canary + macro acc | `tensorboard/fm_<sc>_*` events |
| tables | **B10** velocity + final eyeball diff | control jsons + both CI sets |

**Cut, with reason:** B11 (surprise-vector anecdote) — no artifacts exist on
user_storage (searched `*surprise*`/`*probe*` 2026-07-14); the data was never
persisted from the research era. The draft section reads fine without it.

**Anyscale-post figures live in `blog_figures_anyscale.ipynb`.**


In [ ]:
import os, json, glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

BASE = os.environ.get("FM_BASE", "/mnt/user_storage/transaction-fm-v2")
# Figures default into the REPO (templates/fintech_transaction_fm/figures/) so
# they can be committed and embedded in the blog posts directly; FM_FIG_OUT
# overrides. burst_stats.json / run_costs.json ledgers travel with them.
_repo_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
FIG_OUT = os.environ.get("FM_FIG_OUT", os.path.join(_repo_root, "figures"))
os.makedirs(FIG_OUT, exist_ok=True)

# Palette (validated): one accent for the story (our embedding), muted ink for
# every other model, an ordinal blue ramp for the three context lengths.
# Highlight-one-gray-the-rest; selective direct labels on the accent series.
ACCENT, MUTED = "#2a78d6", "#898781"
INK, INK2 = "#0b0b0b", "#52514e"
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
CTX_RAMP = {512: "#86b6ef", 1024: "#2a78d6", 2048: "#104281"}

SCALES = [("full", 512), ("xl", 1024), ("xxl", 2048)]
NV_BASELINE, NV_FUSION = 0.1238, 0.1755   # their published points

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "axes.labelcolor": INK2, "font.size": 10.5, "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})

def save(fig, name):
    for ext, kw in (("png", {"dpi": 200}), ("svg", {})):
        fig.savefig(f"{FIG_OUT}/{name}.{ext}", bbox_inches="tight", **kw)
    print(f"[saved] {FIG_OUT}/{name}.png + .svg")

def load_results(path):
    if not os.path.exists(path):
        print(f"[SKIP] missing {path}")
        return None
    return json.load(open(path))["results"]

ci100k = {sc: load_results(f"{BASE}/downstream/{sc}/bootstrap_ci.json") for sc, _ in SCALES}
# Table-1 2048 row = the 20-epoch eval (same training budget as 512/1024);
# downstream/xxl holds the 40-ep continuation rerun (table 2's model).
ci100k["xxl"] = (load_results(f"{BASE}/downstream/xxl_old_1783532341/bootstrap_ci.json")
                 or ci100k["xxl"])
cifull = {sc: load_results(f"{BASE}/downstream/{sc}_fulltest/bootstrap_ci.json") for sc, _ in SCALES}

## B1 — hero: two panels, dot + 95% CI

Left = their 100k protocol (the only NVIDIA-comparable set), with their two
published numbers as reference lines. Right = full test period. Shared y-axis;
the caption carries the never-compare-across-panels warning. Our
`embed_xgb` rows wear the accent; every other readout is context.

In [ ]:
def draw_panel(ax, rows, title):
    """rows: list of (label, dict-from-bootstrap_ci, is_ours)."""
    for x, (label, r, ours) in enumerate(rows):
        if r is None:
            continue
        lo, hi = r["ap_ci95"]
        c = ACCENT if ours else MUTED
        ax.errorbar(x, r["ap"], yerr=[[r["ap"] - lo], [hi - r["ap"]]],
                    fmt="o", color=c, ecolor=c, ms=6.5, capsize=3, lw=1.4, zorder=3)
        if ours:  # selective direct labels: only the story series
            ax.annotate(f'{r["ap"]:.3f}', (x, hi), textcoords="offset points",
                        xytext=(0, 5), ha="center", fontsize=9, color=INK)
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels([l for l, _, _ in rows], rotation=22, ha="right",
                       rotation_mode="anchor", fontsize=9)
    ax.set_xlim(-0.6, len(rows) - 0.4)
    ax.set_title(title, fontsize=11, color=INK, pad=10)

if all(ci100k[sc] for sc, _ in SCALES) and all(cifull[sc] for sc, _ in SCALES):
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.4), sharey=True,
                             gridspec_kw={"width_ratios": [7, 4]})

    left = [
        ("raw-13 baseline",    ci100k["full"].get("baseline"),        False),
        ("PCA64 \u00b7 512",  ci100k["full"].get("embed_pca64_xgb"), False),
        ("PCA64 \u00b7 1024", ci100k["xl"].get("embed_pca64_xgb"),   False),
        ("linear \u00b7 512", ci100k["full"].get("embed_logistic"),  False),
        ("XGB \u00b7 512",    ci100k["full"].get("embed_xgb"),       True),
        ("XGB \u00b7 1024",   ci100k["xl"].get("embed_xgb"),         True),
        ("XGB \u00b7 2048 (20ep)", ci100k["xxl"].get("embed_xgb"), True),
    ]
    draw_panel(axes[0], left, "Their protocol \u2014 100k test (112 frauds)")
    axes[0].axhline(NV_FUSION, color=INK2, lw=1.1, ls=(0, (5, 3)), zorder=2)
    axes[0].axhline(NV_BASELINE, color=MUTED, lw=0.9, ls=(0, (5, 3)), zorder=2)
    axes[0].annotate(f"NVIDIA published fusion {NV_FUSION}", (0.985, NV_FUSION),
                     xycoords=("axes fraction", "data"), ha="right", va="bottom",
                     fontsize=8.6, color=INK2,
                     bbox=dict(boxstyle="round,pad=0.15", fc=SURFACE, ec="none", alpha=0.85))
    axes[0].annotate(f"NVIDIA published baseline {NV_BASELINE}", (0.985, NV_BASELINE),
                     xycoords=("axes fraction", "data"), ha="right", va="bottom",
                     fontsize=8.6, color=MUTED,
                     bbox=dict(boxstyle="round,pad=0.15", fc=SURFACE, ec="none", alpha=0.85))
    axes[0].set_ylabel("Average precision (test)")

    right = [
        ("raw-13 baseline",         cifull["full"].get("baseline"),  False),
        ("XGB \u00b7 512",         cifull["full"].get("embed_xgb"), True),
        ("XGB \u00b7 1024",        cifull["xl"].get("embed_xgb"),   True),
        ("XGB \u00b7 2048 (40ep)", cifull["xxl"].get("embed_xgb"),  True),
    ]
    draw_panel(axes[1], right, "Full test period \u2014 2.44M rows (2,724 frauds)")

    # reserve real margins: caption below the tick labels, never through them
    fig.subplots_adjust(bottom=0.26, top=0.80, wspace=0.06)
    fig.suptitle("The embedding alone clears their published fusion headline \u2014 "
                 "and the lift is CI-separated on the full test period",
                 fontsize=12.5, color=INK, y=0.97)
    fig.text(0.5, 0.02, "Dots = point AP, whiskers = bootstrap 95% CI. The two panels are "
             "different eval sets \u2014 never compare numbers across them.",
             ha="center", fontsize=9.5, color=INK2)
    save(fig, "b1_hero")
    plt.show()
else:
    print("[SKIP] B1 \u2014 need bootstrap_ci.json for all scales (100k + fulltest)")


## B2 — pipeline architecture (plain)

The README ASCII diagram, redrawn. The Anyscale-annotated variant is in
`blog_figures_anyscale.ipynb` (it anchors that post).


In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

STAGES = [
    # (engine, title, scale knob, hardware)
    ("Ray Data",  "merchant vocab\n(freq scan, top-K)",      "one column read;\nmerchant_top_k",        "CPU workers"),
    ("Ray Data",  "tokenize\n(static/dynamic, map_groups)",  "seq_len 512\u21922048;\nshuffle_partitions", "CPU-only nodes\n(GPU nodes CPU:0)"),
    ("Ray Train", "masked pretrain + InfoNCE\n(PyTorch DDP)", "num_workers \u00d7 GPU;\nepochs, batch",  "4\u00d7 A10G\n(g5.xlarge)"),
    ("Ray Data",  "embedding extraction\n(CPU read \u2192 GPU forward)", "num_workers=8\n(pure inference)", "0\u21928 GPUs\nautoscaled"),
    ("XGBoost",   "fraud readout\nraw-13 vs embedding",      "xgboost==3.2.0 pin;\nCPU or CUDA",        "head node"),
    ("rank",      "next-merchant reco\n(HR@K / NDCG@K)",     "same frozen backbone,\nInfoNCE head",     "head node"),
    ("Ray Serve", "online embed + score\n(cached)",          "autoscaling\nreplicas",                   "1 GPU replica"),
]
ENGINE_C = {"Ray Data": ACCENT, "Ray Train": "#104281", "XGBoost": "#8a6d1f", "rank": "#8a6d1f", "Ray Serve": "#3d7a4e"}
ARTIFACTS = ["raw parquet\n+ splits.json", "vocab.json", "token windows\n(object store)",
             "checkpoint", "embeddings\nparquet", "metrics +\ntest preds", "", ""]

def draw_arch(annotated: bool, name: str):
    n = len(STAGES)
    fig, ax = plt.subplots(figsize=(14.5, 4.6 if annotated else 3.4))
    ax.set_xlim(-0.75, n - 0.25); ax.set_ylim(-1.9 if annotated else -1.25, 1.35)
    ax.axis("off")
    ax.annotate("raw transactions\n(Parquet, S3)", (-0.62, 0), ha="center", va="center",
                fontsize=9, color=INK2, style="italic")
    for i, (engine, title, knob, hw) in enumerate(STAGES):
        c = ENGINE_C[engine]
        box = FancyBboxPatch((i - 0.36, -0.42), 0.72, 0.9,
                             boxstyle="round,pad=0.02,rounding_size=0.06",
                             fc=SURFACE, ec=c, lw=1.6, zorder=3)
        ax.add_patch(box)
        ax.annotate(engine, (i, 0.30), ha="center", fontsize=9.5, color=c, weight="bold", zorder=4)
        ax.annotate(title, (i, -0.08), ha="center", fontsize=8.2, color=INK, zorder=4)
        if i:  # arrow from previous stage
            ax.add_patch(FancyArrowPatch((i - 1 + 0.40, 0.05), (i - 0.40, 0.05),
                                         arrowstyle="-|>", mutation_scale=13, color=INK2, lw=1.3, zorder=2))
        if ARTIFACTS[i]:  # durable artifact under the boundary
            ax.annotate(ARTIFACTS[i], (i - 0.5 if i else i - 0.62, 0.78), ha="center",
                        fontsize=7.4, color=INK2, zorder=4,
                        bbox=dict(boxstyle="round,pad=0.25", fc=GRID, ec="none", alpha=0.7))
        if annotated:
            ax.annotate(f"scale knob\n{knob}", (i, -0.95), ha="center", va="top", fontsize=7.6, color=INK2)
            ax.annotate(hw, (i, -1.55), ha="center", va="top", fontsize=7.6, color=c)
    ax.annotate("same code laptop \u2192 multi-node: you change the config, not the program",
                (0.5, 0.985), xycoords="axes fraction", ha="center", fontsize=10, color=INK)
    save(fig, name)
    plt.show()

draw_arch(annotated=False, name="b2_architecture")


## B7 — table 2 as a dot-and-CI plot

Every fulltest row; the raw baseline's CI drawn as a reference band so
"CI-separated" is visible rather than asserted. Paired-ordering stats from
`paired_bootstrap_embed_xgb.json` go in the caption.

In [ ]:
pb_path = f"{BASE}/downstream/paired_bootstrap_embed_xgb.json"
pb = json.load(open(pb_path)) if os.path.exists(pb_path) else None

if all(cifull[sc] for sc, _ in SCALES):
    rows = [
        ("PCA64 \u00b7 512",       cifull["full"].get("embed_pca64_xgb"), False),
        ("PCA64 \u00b7 1024",      cifull["xl"].get("embed_pca64_xgb"),   False),
        ("linear \u00b7 512",      cifull["full"].get("embed_logistic"),  False),
        ("linear \u00b7 1024",     cifull["xl"].get("embed_logistic"),    False),
        ("XGB \u00b7 512",         cifull["full"].get("embed_xgb"),       True),
        ("XGB \u00b7 1024",        cifull["xl"].get("embed_xgb"),         True),
        ("XGB \u00b7 2048 (40ep)", cifull["xxl"].get("embed_xgb"),        True),
    ]
    base_r = cifull["full"]["baseline"]

    fig, ax = plt.subplots(figsize=(9.5, 5.4))
    ax.axhspan(*base_r["ap_ci95"], color=GRID, alpha=0.55, zorder=1)
    ax.axhline(base_r["ap"], color=INK2, lw=1.0, zorder=2)
    ax.annotate(f'raw-13 baseline  {base_r["ap"]:.3f} (95% CI band)',
                (0.985, base_r["ap"]), xycoords=("axes fraction", "data"),
                ha="right", va="bottom", fontsize=9, color=INK2)
    draw_panel(ax, rows, "Full test period (2.44M rows): all models on identical rows")
    ax.set_ylabel("Average precision (test)")

    cap = "Dots = point AP, whiskers = bootstrap 95% CI."
    if pb:
        ctx_name = {"full_fulltest": "512", "xl_fulltest": "1024", "xxl_fulltest": "2048"}
        pieces = []
        for k, v in pb["pairs"].items():
            a, b = [ctx_name.get(s.strip(), s.strip()) for s in k.split(" - ")]
            pieces.append(f'{a}\u2212{b}: {v["mean_diff"]:+.3f} '
                          f'[{v["ci95"][0]:+.3f}, {v["ci95"][1]:+.3f}] P={v["p_a_gt_b"]:.3f}')
        cap += "\nPaired bootstrap (same resampled rows): " + " \u00b7 ".join(pieces)
    fig.subplots_adjust(bottom=0.30)
    fig.text(0.5, 0.03, cap, ha="center", fontsize=8.8, color=INK2)
    save(fig, "b7_fulltest")
    plt.show()
else:
    print("[SKIP] B7 \u2014 need fulltest bootstrap_ci.json for all scales")


## B-RECO-FIG — where the FM earns its budget

Grouped bars, HR@10: overall (honest memorization floor vs the hybrid), the
next-merchant∉top-10 slice (memorization structurally 0), and never-seen
merchants (any history method structurally 0). Accent = FM-powered method
(hybrid overall, full-vocab MLP on the slices — labeled per group).

In [ ]:
v3p = f"{BASE}/downstream/full_nextmerchant/probe_metrics_v3.json"
if os.path.exists(v3p):
    v3 = json.load(open(v3p))
    sl = v3["slices"]
    groups = [
        ("Overall\n(400k events)",
         v3["readouts"]["naive_count"]["hr@10"], v3["readouts"]["alpha_blend"]["hr@10"],
         "hybrid: 0.1·MLP + 0.9·freq"),
        (f"Next merchant NOT in card's top-10\n({sl['share_not_in_top10']:.0%} of events)",
         sl["not_in_top10"]["naive_count"], sl["not_in_top10"]["mlp_solo_token_space"],
         "full-vocab MLP"),
        (f"Never-seen merchant\n({sl['share_never_seen_or_beyond_cap']:.0%} of events)",
         sl["never_seen_or_beyond_cap"]["naive_count"],
         sl["never_seen_or_beyond_cap"]["mlp_solo_token_space"], "full-vocab MLP"),
    ]
    x = np.arange(len(groups)); w = 0.36
    fig, ax = plt.subplots(figsize=(9, 4.4))
    bars_naive = ax.bar(x - w / 2, [g[1] for g in groups], w, color=MUTED,
                        edgecolor=SURFACE, linewidth=2,
                        label="memorization baseline (top-10 history)")
    bars_fm = ax.bar(x + w / 2, [g[2] for g in groups], w, color=ACCENT,
                     edgecolor=SURFACE, linewidth=2, label="FM-powered")
    for bars in (bars_naive, bars_fm):
        for rect in bars:
            v = rect.get_height()
            ax.annotate(f"{v:.3f}" if v else "0",
                        (rect.get_x() + rect.get_width() / 2, v),
                        textcoords="offset points", xytext=(0, 4), ha="center",
                        fontsize=9, color=INK)
    for xi, g in zip(x, groups):  # which FM method carries each group
        ax.annotate(g[3], (xi + w / 2, 0), textcoords="offset points", xytext=(0, -34),
                    ha="center", fontsize=8.2, color=INK2)
    ax.set_xticks(x); ax.set_xticklabels([g[0] for g in groups], fontsize=9.5)
    ax.set_ylabel("HR@10"); ax.legend(frameon=False, fontsize=9, loc="upper right")
    ax.set_title("The recommender beats memorization overall — and covers the slices "
                 "memorization is structurally blind to", fontsize=11.5, color=INK, pad=12)
    save(fig, "b_reco_slices")
    plt.show()
else:
    print(f"[SKIP] B-RECO-FIG — missing {v3p}")

## B8 + B9 — the burst stat, computed and ledgered

The draft's "90% of test frauds have a prior same-card fraud within the
preceding 512 transactions vs 7.3% of normals" currently lives in campaign
memory, not an artifact. This computes it from the raw parquet (distance in
same-card transactions to the most recent *strictly prior* fraud), writes
`burst_stats.json` as the citable ledger, and draws the histogram with the
context ceilings (~315 = their token budget, 512/1024/2048 = ours) marked.

In [ ]:
import pyarrow.dataset as pads

raw_p = f"{BASE}/raw/full/transactions.parquet"
splits_p = f"{BASE}/raw/full/splits.json"
if os.path.exists(splits_p):
    splits = json.load(open(splits_p))
    df = (pads.dataset(raw_p, format="parquet")
              .to_table(columns=["card_id", "timestamp", "is_fraud"]).to_pandas())
    df = df.sort_values(["card_id", "timestamp"], kind="stable").reset_index(drop=True)
    is_fraud = df.is_fraud.astype(bool)
    print(f"{len(df):,} transactions, {is_fraud.sum():,} frauds ({is_fraud.mean():.4%})")

    pos = df.groupby("card_id", sort=False).cumcount()
    fraud_pos = pos.where(is_fraud)
    # last fraud at-or-before each row, then shift within card -> strictly prior
    prior = fraud_pos.groupby(df.card_id, sort=False).ffill()
    prior = prior.groupby(df.card_id, sort=False).shift(1)
    dist = pos - prior  # NaN = no prior same-card fraud

    # unit-safe test mask: compare as pandas timestamps, never raw int64
    ts = pd.to_datetime(df["timestamp"])
    cut = pd.to_datetime(splits["val_end"])
    test = ts > cut
    n_test, n_test_fraud = int(test.sum()), int(is_fraud[test].sum())
    print(f"test period (> {cut}): {n_test:,} rows, {n_test_fraud:,} frauds")
    assert n_test > 0 and n_test_fraud > 0, "test mask is empty — check splits val_end vs timestamp dtype"

    stats = {"definition": "distance in same-card transactions to most recent "
                           "strictly-prior fraud; test-period rows (ts > val_end)",
             "n_test": n_test, "n_test_fraud": n_test_fraud}
    groups = {}
    for grp_name, mask in (("fraud", test & is_fraud), ("normal", test & ~is_fraud)):
        d_grp = dist[mask]
        groups[grp_name] = d_grp
        s = {"n": int(mask.sum()), "share_no_prior_fraud": float(d_grp.isna().mean())}
        for k in (315, 512, 1024, 2048):
            s[f"share_within_{k}"] = float((d_grp <= k).mean())
        stats[grp_name] = s
        print(f"  {grp_name}: n={s['n']:,}  within 512 = {s['share_within_512']:.1%}  "
              f"no prior fraud = {s['share_no_prior_fraud']:.1%}")
    with open(f"{FIG_OUT}/burst_stats.json", "w") as f:
        json.dump(stats, f, indent=2)
    print(f"[saved] {FIG_OUT}/burst_stats.json  <- cite these exact numbers in B8")

    # Cumulative form: share of the group whose most recent same-card fraud is
    # within x transactions — the 90%-vs-7.3% claim is read straight off x=512.
    fig, ax = plt.subplots(figsize=(9.5, 5))
    xmax = max(float(dist.max()), 4096.0)
    for name, key, color in (("fraud txns", "fraud", ACCENT), ("normal txns", "normal", MUTED)):
        d_grp = groups[key].dropna().sort_values().to_numpy()
        n_grp = stats[key]["n"]  # denominator includes no-prior rows
        xs = np.concatenate([[1.0], d_grp.clip(min=1.0), [xmax]])
        ys = np.concatenate([[0.0], (np.arange(len(d_grp)) + 1) / n_grp,
                             [len(d_grp) / n_grp]])
        ax.plot(xs, ys, color=color, lw=2.2, drawstyle="steps-post", zorder=3)
        at512 = stats[key]["share_within_512"]
        # anchored label positions clear of both curves (fraud rides ~90%,
        # normal hugs 0 on the left) instead of data-driven spots on the line
        lx, ly = (0.30, 0.70) if key == "fraud" else (0.30, 0.14)
        ax.annotate(f"{name}: {at512:.0%} within 512", (lx, ly),
                    xycoords="axes fraction", fontsize=10, color=color,
                    weight="bold", zorder=4)
    for k, lab, ha in ((315, "~315 (their budget) ", "right"), (512, " 512", "left"),
                       (1024, "1024", "center"), (2048, "2048", "center")):
        ax.axvline(k, color=AXIS, lw=0.9, ls=(0, (4, 3)), zorder=2)
        ax.annotate(lab, (k, 1.02), ha=ha, va="bottom", fontsize=8, color=INK2,
                    annotation_clip=False)
    ax.set_xscale("log")
    ax.set_xlim(1, xmax); ax.set_ylim(0, 1.0)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0%", "25%", "50%", "75%", "100%"])
    ax.set_xlabel("same-card transactions since previous fraud (log scale)")
    ax.set_ylabel("share of group with a prior fraud this recent")
    ax.set_title("Fraud is bursty: the signal sits within the context window",
                 fontsize=11.5, color=INK, pad=28)
    fig.subplots_adjust(top=0.85)
    save(fig, "b9_burst")
    plt.show()
else:
    print(f"[SKIP] B8/B9 — missing {splits_p}")


## B16 — the month canary + macro accuracy

`field_ce/month` per optimizer step (same optimizer work = fair x-axis across
context lengths) and `epoch/acc_macro` per epoch, all three scales on the
validated ordinal blue ramp. The 2048 run's 20→40 continuation appears
naturally (its resumed run carries both event streams).

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def scalars(scale, tag):
    """Merge a tag across every event file/run dir for a scale, sorted by step."""
    pts = []
    for run in sorted(glob.glob(f"{BASE}/tensorboard/fm_{scale}_*")):
        acc = EventAccumulator(run); acc.Reload()
        if tag in acc.Tags()["scalars"]:
            pts += [(e.step, e.value) for e in acc.Scalars(tag)]
    return sorted(set(pts)) if pts else None

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
found = False
for scale, ctx in SCALES:
    for ax, tag in ((axes[0], "field_ce/month"), (axes[1], "epoch/acc_macro")):
        pts = scalars(scale, tag)
        if pts:
            found = True
            xs, ys = zip(*pts)
            ax.plot(xs, ys, color=CTX_RAMP[ctx], lw=2, label=f"{ctx} ctx", zorder=3)
            ax.annotate(f"{ctx}", (xs[-1], ys[-1]), textcoords="offset points",
                        xytext=(5, 0), fontsize=9, color=CTX_RAMP[ctx], va="center")
if found:
    axes[0].set_title("month canary — field_ce/month (lower = learned)",
                      fontsize=11, color=INK)
    axes[0].set_xlabel("optimizer step"); axes[0].set_ylabel("cross-entropy")
    axes[1].set_title("per-field macro accuracy by epoch", fontsize=11, color=INK)
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy")
    axes[0].legend(frameon=False, fontsize=9)
    fig.suptitle("Longer context = fewer windows per epoch: the 512 run hits its month "
                 "phase transition; the longer runs grind toward theirs",
                 fontsize=12, color=INK, y=1.03)
    save(fig, "b16_canary")
    plt.show()
else:
    print(f"[SKIP] B16 — no event files under {BASE}/tensorboard/fm_*")

## B10 + final eyeball diff

The velocity-control numbers for the B10 table, the shuffled-label control,
and both CI tables printed once more — diff these against the draft's inline
tables *before* exporting anything above, so figures and prose can't drift.

In [ ]:
for p, label in (
        (f"{BASE}/downstream/full_velocity/velocity_metrics.json", "B10 velocity control"),
        (f"{BASE}/downstream/full_probe/probe_metrics_shuffled_seed0.json",
         "shuffled-label control")):
    if os.path.exists(p):
        print(f"== {label}: {p}")
        print(json.dumps(json.load(open(p)), indent=2)[:2000])
    else:
        print(f"[SKIP] {label} — missing {p}")

for name, src in (("TABLE 1 (100k)", ci100k), ("TABLE 2 (fulltest)", cifull)):
    rows = []
    for sc, ctx in SCALES:
        if src[sc]:
            for m, r in src[sc].items():
                rows.append({"model": m, "ctx": ctx, "ap": round(r["ap"], 4),
                             "ci": [round(x, 3) for x in r["ap_ci95"]],
                             "P(>fusion)": round(r.get("p_beats_nvidia_fusion",
                                                       float("nan")), 3)})
    if rows:
        print(f"\n== {name} — diff against BLOG_DRAFT.md")
        print(pd.DataFrame(rows).to_string(index=False))

---
**Produced here:** `b1_hero`, `b2_architecture`, `b7_fulltest`, `b_reco_slices`,
`b9_burst` (+ `burst_stats.json`, the B8 ledger), `b16_canary` — PNG+SVG each.

**Still manual:** Zach sign-offs + links, venue/CTA decisions. B11 cut (no data).
**Anyscale post:** `blog_figures_anyscale.ipynb`.
